# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [22]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _line item_

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [23]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'], sep='.', errors="ignore")

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-08


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [24]:
# TODO
vendors = pd.DataFrame(vendors_json)
# left join
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
joined

# V-08 did not match, and it has 3 units attached to it.

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [25]:
# Assign it a placeholder name to allow calculations with the item even if it's not in the vendor list
joined['name'] = joined['name'].fillna('Unknown Vendor')
joined['zone'] = joined['zone'].fillna('Unknown Zone')

joined

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,Unknown Vendor,Unknown Zone,left_only


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [26]:
# check which zone sold the most items
zone_totals = joined.groupby('zone')['qty'].sum().reset_index()
zone_totals = zone_totals.sort_values(by='qty', ascending=False)
zone_totals

# 7 items, with zone A selling the most, and the rest going to the unknown zone

,zone,qty
0,A,4
1,Unknown Zone,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [27]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

ragged_rows = [
    {
        'order': order['order'],
        'vendor_id': order['vendor_id'],
        'item': line.get('item'),
        'qty': line.get('qty')
    }
    for order in ragged_json
        for line in order.get('lines', [])
]

ragged_lines = pd.DataFrame(ragged_rows)
ragged_lines

,order,vendor_id,item,qty
0,4,V-01,Soda,2.0
1,6,V-01,Fries,NaN


### Q6 — Validate

In [28]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) Line items are the innermost unit. It would be harder with one row per order because then you'd have multiple line items in a single cell.

b) A missing quantity indicates that the data is unavailable, and zero means there are no items of that type. Conflating them would lead to incorrect analysis and reporting (cannot distinguish between true zero and a missing value)